# ___Decide the discrete character transition model based on the supplementary data of `Maherali, H. et al. (2016)`___
-------------------

In [1]:
# ‘Mutualism Persistence and Abandonment during the Evolution of the Mycorrhizal Symbiosis’, The American Naturalist, 188(5), pp. E113–E125. Available at: https://doi.org/10.1086/688675.

In [3]:
suppressPackageStartupMessages({
    library("ape")
    library("geiger")
    library("phytools")
    library("corHMM")
    library("U.PhyloMaker")
})

In [55]:
states <- read.csv("../../data/chapter2/Maherali.etal.AmNat.Data.csv") # supplementary dataset from the paper
colnames(states) <- c("binominal", "state")
head(states)

,binominal,state
,<chr>,<chr>
1,Abies_alba,EM
2,Abies_amabilis,EM
3,Abies_cephalonica,EM
4,Abies_concolor,EM
5,Abies_firma,EM
6,Abies_fraseri,EM


In [56]:
# now we need a phylogeny for all these species

megatree <- ape::read.tree("../../data/chapter2/uphylomaker/GBOTB.extended.TPL.tre") # the TPL megatree

In [57]:
# species list must be a dataframe with following columns
# species,genus,family,species.relative,genus.relative

# gsub(states$Genus_species, pattern = '_', replacement = ' ')
# gsub(states$Genus_species, pattern = "_\\w+", replacement = '')

species_list <- data.frame(species = gsub(states$binominal, pattern = '_', replacement = ' '),
              genus = gsub(states$binominal, pattern = "_\\w+", replacement = ''), family = NA, species.relative = NA, genus.relative = NA)

head(species_list) # good

,species,genus,family,species.relative,genus.relative
,<chr>,<chr>,<lgl>,<lgl>,<lgl>
1,Abies alba,Abies,NA,NA,NA
2,Abies amabilis,Abies,NA,NA,NA
3,Abies cephalonica,Abies,NA,NA,NA
4,Abies concolor,Abies,NA,NA,NA
5,Abies firma,Abies,NA,NA,NA
6,Abies fraseri,Abies,NA,NA,NA


In [58]:
unique_genera <- unique(species_list$genus)
length(unique_genera)

[1] 1337

In [59]:
# genus list needs to be a dataframe with the following columns 
# genus,family

taxonlookup <- read.csv("../../data/chapter2/plantlookup_serialized_from_r.csv")

In [60]:
intersect(unique_genera, taxonlookup$genus) |> length() # not bad at all

[1] 1333

In [61]:
species_list_pruned <- species_list[species_list$genus %in% taxonlookup$genus, ] # drop the genera that do not have family info in taxonlookup
stopifnot(length(unique(species_list_pruned$genus))==length(intersect(unique_genera, taxonlookup$genus)))

genus_list <- taxonlookup[taxonlookup$genus %in% unique(species_list_pruned$genus), ] # subset the taxonlookup dataset to only include the genus that we have mycorrhizal state data for
stopifnot(all(genus_list$genus %in% unique(species_list_pruned$genus)))

In [62]:
phylogeny <- U.PhyloMaker::phylo.maker(sp.list = species_list_pruned, tree = megatree, gen.list = genus_list) # phylogenetic tree creation

[1] "Note: 1 species fail to be binded to the tree."
[1] "Musa_acuminata"


In [63]:
ape::write.tree(phy = phylogeny$phylo, file = "../../data/chapter2/uphylomaker/maherali_2016.tre")

In [64]:
phylogeny <- ape::read.tree("../../data/chapter2/uphylomaker/maherali_2016.tre")
phylogeny <- ape::multi2di(phylogeny) # make the tree binary

stopifnot(ape::is.binary(phylogeny))

In [65]:
sum(phylogeny$edge.length == 0) # damn

[1] 257

In [78]:
states_named_vector <- setNames(states$state, nm = states$binominal) # for phytools and ape
states_named_vector <- states_named_vector[-which(names(states_named_vector) == "Musa_acuminata")] # drop the species that failed to bind to the megatree during the phylogeny creation

In [82]:
which(names(states_named_vector) == "Musa_acuminata")

integer(0)

In [79]:
tm <- Sys.time()

hmm_er <- corHMM::corHMM(phy = phylogeny, data = states, model = "ER", node.states = "marginal", rate.cat = 1)
hmm_sym <- corHMM::corHMM(phy = phylogeny, data = states, model = "SYM", node.states = "marginal", rate.cat = 1)
hmm_ard <- corHMM::corHMM(phy = phylogeny, data = states, model = "ARD", node.states = "marginal", rate.cat = 1)

ape_er <- ape::ace(phy = phylogeny, x = states_named_vector, type = "discrete", method = "ML", model = "ER", marginal = FALSE)
ape_sym <- ape::ace(phy = phylogeny, x = states_named_vector, type = "discrete", method = "ML", model = "SYM", marginal = FALSE)
ape_ard <- ape::ace(phy = phylogeny, x = states_named_vector, type = "discrete", method = "ML", model = "ARD", marginal = FALSE)

ancr_er <- phytools::ancr(phytools::fitMk(tree = phylogeny, x = states_named_vector, model = "ER"))
ancr_sym <- phytools::ancr(phytools::fitMk(tree = phylogeny, x = states_named_vector, model = "SYM"))
ancr_ard <- phytools::ancr(phytools::fitMk(tree = phylogeny, x = states_named_vector, model = "ARD"))

tm <- Sys.time() - tm

save(hmm_er, hmm_sym, hmm_ard, ape_er, ape_sym, ape_ard, ancr_er, ancr_sym, ancr_ard, file = "./../../data/chapter2/rdata/ace_states_maherali_2016.RData")

You specified 'fixed.nodes=FALSE' but included a phy object with node labels. These node labels have been removed.


Warning message in corHMM::corHMM(phy = phylogeny, data = states, model = "ER", :
"Branch lengths of 0 detected. Adding 1e-5 to these branches."


State distribution in data:
States:	1	2	3	4	
Counts:	1753	521	270	429	
Beginning thorough optimization search -- performing 0 random restarts 
Finished. Inferring ancestral states using marginal reconstruction. 
You specified 'fixed.nodes=FALSE' but included a phy object with node labels. These node labels have been removed.


Warning message in corHMM::corHMM(phy = phylogeny, data = states, model = "SYM", :
"Branch lengths of 0 detected. Adding 1e-5 to these branches."


State distribution in data:
States:	1	2	3	4	
Counts:	1753	521	270	429	
Beginning thorough optimization search -- performing 0 random restarts 
Finished. Inferring ancestral states using marginal reconstruction. 
You specified 'fixed.nodes=FALSE' but included a phy object with node labels. These node labels have been removed.


Warning message in corHMM::corHMM(phy = phylogeny, data = states, model = "ARD", :
"Branch lengths of 0 detected. Adding 1e-5 to these branches."


State distribution in data:
States:	1	2	3	4	
Counts:	1753	521	270	429	
Beginning thorough optimization search -- performing 0 random restarts 
Finished. Inferring ancestral states using marginal reconstruction. 


ERROR: Error in ape::ace(phy = phylogeny, x = states_named_vector, type = "discrete", : length of phenotypic and of phylogenetic data do not match.


In [ ]:
load("../../data/chapter2/rdata/ace_states_maherali_2016.RData")

corhmm_ace <- list(ER=hmm_er, SYM=hmm_sym, ARD=hmm_ard)
ape_ace <- list(ER=ape_er, SYM=ape_sym, ARD=ape_ard)
phytools_ace <- list(ER=ancr_er, SYM=ancr_sym, ARD=ancr_ard)

In [ ]:
# log likelihoods (lnLik)
data.frame(ape = mapply(ape_ace, FUN = function(model) model$loglik), corHMM = mapply(corhmm_ace, FUN = function(model) model$loglik), phytools = mapply(phytools_ace, FUN = function(model) model$logLik))

In [ ]:
# AIC
data.frame(ape = mapply(ape_ace, FUN = stats::AIC), phytools = mapply(phytools_ace, FUN = stats::AIC), corHMM = mapply(corhmm_ace, FUN = function(model) model$AIC)) # stats::AIC() fails for corHMM models